- Split VQA-RAD data into images/, train.json, val.json, test.json

In [ ]:
import os
import json
import shutils
import random
from collections import defaultdict

In [ ]:
# Configuration

RAW_JSON = "data/VQA_RAD Dataset Public.json"    # Original annotations
RAW_IMAGE_DIR = "data/VQA_RAD Image Folder"      # Original images folder
OUTPUT_DIR = "VQA_RAD"                           # Final destination folder

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15 
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

In [5]:
# Output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "images"), exist_ok=True)

In [8]:
# Load annotations
with open(RAW_JSON, "r") as f:
    data = json.load(f)

print(f"Total samples: {len(data)}")

Total samples: 2248


In [11]:
# Group samples by image 
# Ensure the same image does not appear in different splits
image_to_samples = defaultdict(list)
for item in data:
    image_to_samples[item["image_name"]].append(item)

image_names = list(image_to_samples.keys())
random.shuffle(image_names)

# Compute split sizes
num_images = len(image_names)
train_end = int(TRAIN_RATIO * num_images)
val_end = train_end + int(VAL_RATIO * num_images)

train_images = image_names[:train_end]
val_images = image_names[train_end:val_end]
test_images = image_names[val_end:]

print(f"Train images: {len(train_images)}")
print(f"Val images: {len(val_images)}")
print(f"Test images: {len(test_images)}")

Train images: 219
Val images: 47
Test images: 48


In [12]:
# Build split annotation files
def build_split(image_list):
    split_data = []
    for img in image_list:
        split_data.extend(image_to_samples[img])
    return split_data

train_data = build_split(train_images)
val_data = build_split(val_images)
test_data = build_split(test_images)

In [14]:
# Save JSON file
with open(os.path.join(OUTPUT_DIR, "train.json"), "w") as f:
    json.dump(train_data, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "val.json"), "w") as f:
    json.dump(val_data, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "test.json"), "w") as f:
    json.dump(test_data, f, indent=2)

In [15]:
# Copy images (Once)
used_images = set(train_images + val_images + test_images)

for img_name in used_images:
    src = os.path.join(RAW_IMAGE_DIR, img_name)
    dst = os.path.join(OUTPUT_DIR, "images", img_name)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

print("Dataset split completed successfully!")

Dataset split completed successfully!
